# 02. Posit, Quire, Takum, and Logarithmic Number Systems (LNS)

- **Category:** 13_precision_and_numerics
- **Lineage & Citations:** Kingsbury & Rayner (1971 *Digital filtering using logarithmic arithmetic*); Kulisch (1981 *Computer Arithmetic in Theory and Practice*); Gustafson (2017 *The End of Error: Unum Computation & Posits*).
- **Purpose:** Formulate and evaluate non-IEEE numerical architectures: Posit tapered precision arithmetic in the "Golden Zone", the Kulisch Quire exact dot-product accumulator, and multiplier-free Logarithmic Number System (LNS) arithmetic.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [01. Floating Point Anatomy Subnormal Traps and Stochastic Logic](01_floating_point_anatomy_subnormal_traps_and_stochastic_logic.ipynb) $\to` `🔗 Downstream Syntheses:` [03. Automatic Mixed Precision Autocast and GradScaler](03_automatic_mixed_precision_autocast_and_gradscaler.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not select large Posit exponent scales ($es \ge 3$ in 8-bit) when weights cluster near $\pm 1$ (erodes fraction bits below standard FP8); do not use uncorrected Mitchell's approximation in deep LNS networks (accumulates positive bias).

## Mathematical Foundations & Non-IEEE Formulations

1. **Posit Arithmetic Representation**:
   $$x = (-1)^s \times \text{useed}^k \times 2^e \times (1 + f), \quad \text{useed} = 2^{2^{es}}$$
2. **Logarithmic Number System (LNS) Multiplication**:
   $$x \cdot y = (-1)^{s_x \oplus s_y} \cdot 2^{L_x + L_y}, \quad L = \log_2(|x|)$$
3. **LNS Gaussian Log-Addition**:
   $$L_{a+b} = \max(L_a, L_b) + \log_2(1 + 2^{-|L_a - L_b|})$$
4. **Kulisch Quire Exact Accumulator**:
   $$Q = \sum_{i=1}^M a_i b_i \quad \text{(Zero intermediate rounding error)}$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/13_precision_and_numerics/02_posit_quire_takum_and_logarithmic_number_systems_lns.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def posit_tapered_quantize(x, n_bits=8, es=1):
    is_zero = (x == 0.0)
    x_safe = torch.where(is_zero, torch.ones_like(x), torch.abs(x))
    log2_x = torch.floor(torch.log2(x_safe))
    useed = 2.0 ** float(es)
    k = torch.floor(log2_x / useed)
    regime_bits = torch.where(k >= 0.0, k + 2.0, -k + 1.0)
    frac_bits = torch.clamp(float(n_bits) - 1.0 - regime_bits - float(es), min=0.0, max=float(n_bits - 2))
    step = 2.0 ** (log2_x - frac_bits)
    q = torch.round(x_safe / step) * step
    return torch.where(is_zero, torch.zeros_like(x), torch.sign(x) * q)

def lns_multiplication(a, b, eps=1e-12):
    log_out = torch.log2(torch.abs(a) + float(eps)) + torch.log2(torch.abs(b) + float(eps))
    return torch.sign(a) * torch.sign(b) * (2.0 ** log_out)

def lns_gaussian_addition(a, b, eps=1e-12):
    la, lb = torch.log2(torch.clamp(a, min=float(eps))), torch.log2(torch.clamp(b, min=float(eps)))
    return 2.0 ** (torch.maximum(la, lb) + torch.log2(1.0 + (2.0 ** (-torch.abs(la - lb)))))

def quire_dot_product_sim(a, b):
    return torch.sum(a.double() * b.double(), dim=-1).float()

In [ ]:
# Verify Posit, LNS, and Quire against golden reference tensors
posit = posit_tapered_quantize(weights["posit_in"], n_bits=8, es=1)
assert torch.max(torch.abs(posit - weights["posit_ref"])) < 1e-5

lns_mul = lns_multiplication(weights["lns_a"], weights["lns_b"])
assert torch.max(torch.abs(lns_mul - weights["lns_mul_ref"])) < 1e-5

lns_add = lns_gaussian_addition(weights["add_a"], weights["add_b"])
assert torch.max(torch.abs(lns_add - weights["lns_add_ref"])) < 1e-5

quire = quire_dot_product_sim(weights["quire_mat"], weights["quire_vec"])
assert torch.max(torch.abs(quire - weights["quire_ref"])) < 1e-5
print("All Posit, LNS, and Quire parity assertions verified!")

In [ ]:
# Microbenchmark: Posit vs LNS multiplication latency
a, b = weights["lns_a"], weights["lns_b"]
t0 = time.perf_counter()
for _ in range(500):
    _ = lns_multiplication(a, b)
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"LNS multiplication latency: {lat_us:.2f} us")

In [ ]:
# Golden Zone Verification: Posit precision near 1.0
near_one = torch.tensor([0.875, 1.0, 1.125, 1.25, 1.5, 1.75])
posit_out = posit_tapered_quantize(near_one, n_bits=8, es=1)
err_golden_zone = float(torch.max(torch.abs(near_one - posit_out)))
print(f"Posit8 max error in Golden Zone [0.8, 1.8]: {err_golden_zone:.2e}")
assert err_golden_zone < 1e-5, "Posit8 must have exact representation in Golden Zone!"

# Quire Exactness Check: Prevent catastrophic cancellation
v_large = torch.tensor([1e8, -1e8, 3.14159])
v_unit = torch.tensor([1.0, 1.0, 1.0])
q_res = quire_dot_product_sim(v_large.unsqueeze(0), v_unit).item()
assert abs(q_res - 3.14159) < 1e-4, f"Quire failed exact cancellation: {q_res}"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Posits** concentrate fraction precision in the "Golden Zone" near $\pm 1$, providing superior Signal-to-Quantization-Noise Ratio (SQNR) for normalized neural weights.
2. **The Kulisch Quire** eliminates intermediate rounding errors in matrix operations via wide fixed-point registers, preventing catastrophic cancellation.
3. **LNS Arithmetic** converts expensive multiplication into hardware-cheap fixed-point addition, using logarithmic correction tables for addition.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [02_posit_quire_takum_and_logarithmic_number_systems_lns.md](../../markdown/13_precision_and_numerics/02_posit_quire_takum_and_logarithmic_number_systems_lns.md)
- **PyTorch 2.x Module:** [02_posit_quire_takum_and_logarithmic_number_systems_lns.py](../../pytorch2/13_precision_and_numerics/02_posit_quire_takum_and_logarithmic_number_systems_lns.py)
- **Functional JAX Module:** [02_posit_quire_takum_and_logarithmic_number_systems_lns.py](../../jax/13_precision_and_numerics/02_posit_quire_takum_and_logarithmic_number_systems_lns.py)
- **Native Rust Candle Track:** [02_posit_quire_takum_and_logarithmic_number_systems_lns.rs](../../candle/13_precision_and_numerics/02_posit_quire_takum_and_logarithmic_number_systems_lns.rs)
- **Hardware Kernel (CPU / WGSL):** [02_posit_quire_takum_and_logarithmic_number_systems_lns.rs](../../kernels/13_precision_and_numerics/02_posit_quire_takum_and_logarithmic_number_systems_lns.rs)
- **Golden SafeTensors:** [02_posit_quire_takum_and_logarithmic_number_systems_lns.safetensors](../../weights/13_precision_and_numerics/02_posit_quire_takum_and_logarithmic_number_systems_lns.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 01. Floating Point Anatomy Subnormal Traps and Stochastic Logic](01_floating_point_anatomy_subnormal_traps_and_stochastic_logic.ipynb) | [Master Index](../../README.md#13_precision_and_numerics) | [Next: 03. Automatic Mixed Precision Autocast and GradScaler](03_automatic_mixed_precision_autocast_and_gradscaler.ipynb) &rarr;